In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ---------------- CONFIG ----------------
RUN_MODELS = ["distilbert-base-uncased"]
RUN_TASKS = ["cola","mrpc"]
K_LIST = [10, 8, 6, 4, 3, 2]      # retained heads per layer (of 12)  -> 83,67,50,33,25,17 %
N_RANDOM = 5                       # random draws per budget
GRAD_N = 512                       # dev examples for gradient importance (batch size 1)
ABL_N = 1500                       # dev examples for single-head ablation
KAPPA_Z = 1.0                      # LA rule (fixed in advance): protect layers with mean kappa > mean + KAPPA_Z*SD
METHODS = None                     # None = run every method. Or a list, e.g. ["asicr_inv_uniform","frob_inv_uniform","chain_inv_uniform"]
RUN_ABLATION = True                # set False when re-running only extra METHODS
OUT_DIR = "/kaggle/working/prune_results"
SEARCH = [
    "/kaggle/input/notebooks/*/*/ckpt/*/done.json",   
    "/kaggle/input/*/ckpt/*/done.json",               
    "/kaggle/input/**/done.json",                     
    "/kaggle/working/ckpt/**/done.json",              
]
MAX_LEN = 128

import glob, json, time
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr


def uniform_keep(score, k):
    """Keep the k highest-score heads in every layer. score: (L,H) -> bool (L,H)."""
    L, H = score.shape
    keep = np.zeros((L, H), bool)
    for l in range(L):
        keep[l, np.argsort(-score[l], kind="stable")[:k]] = True
    return keep


def global_keep(score, N):
    """Keep the N highest-score heads over the whole network."""
    L, H = score.shape
    keep = np.zeros(L * H, bool)
    keep[np.argsort(-score.ravel(), kind="stable")[:N]] = True
    return keep.reshape(L, H)


def la_keep(score, k, protect):
    """Layer-adaptive, BUDGET-MATCHED: protected layers keep all heads; the remaining
    budget N-|P|*H is spread over unprotected layers. Total kept == k*L exactly.
    Returns None if the budget cannot cover the protected layers."""
    L, H = score.shape
    N = k * L
    P = sorted(set(protect))
    rem = N - len(P) * H
    if rem < 0:
        return None
    unprot = [l for l in range(L) if l not in P]
    keep = np.zeros((L, H), bool)
    for l in P:
        keep[l, :] = True
    if unprot:
        base, extra = divmod(rem, len(unprot))
        order = sorted(unprot, key=lambda l: -score[l].mean())     # extras to highest-norm layers
        for i, l in enumerate(order):
            kk = base + (1 if i < extra else 0)
            keep[l, np.argsort(-score[l], kind="stable")[:kk]] = True
    elif rem > 0:
        return None
    assert keep.sum() == N, (keep.sum(), N)
    return keep


def kappa_protect(kappa, z=1.0):
    """Layers whose mean kappa exceeds (mean of layer means) + z * SD of layer means."""
    lm = kappa.mean(1)
    thr = lm.mean() + z * lm.std()
    return [int(l) for l in np.where(lm > thr)[0]], float(thr)


import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import matthews_corrcoef, f1_score

dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEYS = {"sst2": ("sentence", None), "cola": ("sentence", None), "mrpc": ("sentence1", "sentence2")}


def get_attn(model):
    """list of (q, k, v, out) nn.Linear per layer; n_heads; hidden."""
    if hasattr(model, "distilbert"):
        ls = model.distilbert.transformer.layer
        acc = [(l.attention.q_lin, l.attention.k_lin, l.attention.v_lin, l.attention.out_lin) for l in ls]
        return acc, model.config.n_heads, model.config.dim
    ls = model.bert.encoder.layer
    acc = [(l.attention.self.query, l.attention.self.key, l.attention.self.value, l.attention.output.dense) for l in ls]
    return acc, model.config.num_attention_heads, model.config.hidden_size


def prep(task, tok):
    ds = load_dataset("nyu-mll/glue", task)
    k1, k2 = KEYS[task]

    def f(b):
        return tok(b[k1], b[k2], truncation=True, max_length=MAX_LEN) if k2 else tok(b[k1], truncation=True, max_length=MAX_LEN)

    split = ds["train"].train_test_split(test_size=0.05, seed=0)          # identical to fine-tuning script
    parts = {"train": split["train"], "dev": split["test"], "val": ds["validation"]}
    cols = ["input_ids", "attention_mask", "label"]
    out = {}
    for n, d in parts.items():
        d = d.map(f, batched=True)
        out[n] = d.remove_columns([c for c in d.column_names if c not in cols])
    n_sub = min(5000, len(out["train"]))
    idx = np.random.RandomState(0).choice(len(out["train"]), n_sub, replace=False)
    out["train_sub"] = out["train"].select(sorted(idx.tolist()))
    out["dev_abl"] = out["dev"].select(range(min(ABL_N, len(out["dev"]))))
    out["dev_grad"] = out["dev"].select(range(min(GRAD_N, len(out["dev"]))))
    return out


def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b["label"] for b in batch])
        enc = tok.pad([{k: b[k] for k in ("input_ids", "attention_mask")} for b in batch], return_tensors="pt")
        enc["labels"] = labels
        return enc
    return collate


@torch.no_grad()
def predict(model, ds, collate, bs=128):
    model.eval()
    lg, lb = [], []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        lab = b.pop("labels")
        b = {k: v.to(dev_) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            lg.append(model(**b).logits.float().cpu())
        lb.append(lab)
    return torch.cat(lg).numpy(), torch.cat(lb).numpy()


def metrics(task, logits, labels):
    p = logits.argmax(1)
    loss = torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()
    m = {"acc": float((p == labels).mean()), "loss": loss}
    m["mcc"] = float(matthews_corrcoef(labels, p)) if task == "cola" else np.nan
    m["f1"] = float(f1_score(labels, p)) if task == "mrpc" else np.nan
    return m


def head_scores(model):
    """All per-head scores from the DENSE weights. Returns dict of (L,H) arrays + timing."""
    acc, H, D = get_attn(model)
    dh = D // H
    L = len(acc)
    S = {k: np.zeros((L, H)) for k in ["frob", "op", "chain", "kappa", "kappa_row"]}
    t0 = time.time()
    with torch.no_grad():
        for l, (q, k, v, o) in enumerate(acc):
            Wq, Wk, Wv, Wo = [m.weight.detach().float().cpu() for m in (q, k, v, o)]
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                Woh = Wo[:, sl]                                   # (D, dh)  rows = hidden dims
                sv = torch.linalg.svdvals(Woh)
                S["op"][l, h] = sv[0].item()
                S["frob"][l, h] = torch.linalg.matrix_norm(Woh, "fro").item()
                S["kappa"][l, h] = (sv[0] ** 2 / (sv ** 2).mean()).item()   # sigma1^2 / mean sigma_i^2  (= rho_h^2)
                rn = (Woh ** 2).sum(1)
                S["kappa_row"][l, h] = (rn.max() / (rn.mean() + 1e-12)).item()   # old notebook's "kappa"
                bq = torch.linalg.matrix_norm(Wq[sl, :], 2).item()
                bk = torch.linalg.matrix_norm(Wk[sl, :], 2).item()
                bv = torch.linalg.matrix_norm(Wv[sl, :], 2).item()
                S["chain"][l, h] = bq * bk * bv * sv[0].item()
    S["_time_svd_s"] = time.time() - t0
    return S


def grad_importance(model, ds, collate):
    """Michel et al. (2019)-style: E_x |dL/dxi_h| with xi = per-head gate on the out-projection input."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    xi = torch.ones(L, H, device=dev_, requires_grad=True)
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0]
            B, T, Dm = x.shape
            return ((x.reshape(B, T, H, dh) * xi[l].view(1, 1, H, 1)).reshape(B, T, Dm),)
        hooks.append(o.register_forward_pre_hook(pre))
    imp = torch.zeros(L, H, device=dev_)
    model.eval()
    t0 = time.time()
    for b in DataLoader(ds, batch_size=1, collate_fn=collate):
        b = {k: v.to(dev_) for k, v in b.items()}
        loss = model(**b).loss
        g, = torch.autograd.grad(loss, xi)
        imp += g.abs()
    for h_ in hooks:
        h_.remove()
    return (imp / len(ds)).detach().cpu().numpy(), time.time() - t0


class Pruner:
    """Applies a keep-mask by zeroing the out-projection columns of pruned heads (exactly removes the head)."""
    def __init__(self, model):
        self.model = model
        self.acc, self.H, self.D = get_attn(model)
        self.dh = self.D // self.H
        self.W0 = [o.weight.detach().clone() for (_, _, _, o) in self.acc]

    def apply(self, keep):
        with torch.no_grad():
            for l, (_, _, _, o) in enumerate(self.acc):
                o.weight.copy_(self.W0[l])
                for h in range(self.H):
                    if not keep[l, h]:
                        o.weight[:, h * self.dh:(h + 1) * self.dh] = 0.0


def discover():
    runs = {}
    for pat in SEARCH:
        for p in glob.glob(pat, recursive=True):
            try:
                r = json.load(open(p))
            except Exception:
                continue
            runs[(r["model"], r["task"], r["seed"])] = os.path.dirname(p)
    sel = {k: v for k, v in runs.items() if k[0] in RUN_MODELS and k[1] in RUN_TASKS}
    if not sel:
        print("No matching checkpoints. All done.json files seen:", sorted(runs.keys()))
        print("Top of /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count(os.sep) <= 5:
                print("  ", root, dirs[:6])
    return dict(sorted(sel.items()))


def run_checkpoint(key, path, cache):
    mname, task, seed = key
    tag = f"{mname}_{task}_seed{seed}"
    odir = f"{OUT_DIR}/{tag}"
    if os.path.exists(f"{odir}/DONE"):
        print("skip (done):", tag); return
    os.makedirs(odir, exist_ok=True)
    if (mname, task) not in cache:
        tok = AutoTokenizer.from_pretrained(path)
        cache[(mname, task)] = (tok, prep(task, tok), make_collate(tok))
    tok, data, collate = cache[(mname, task)]
    model = AutoModelForSequenceClassification.from_pretrained(path).to(dev_).eval()
    pr = Pruner(model)
    L, H = len(pr.acc), pr.H
    print(f"\n=== {tag}: L={L} H={H}")

    S = head_scores(model)
    G, t_grad = grad_importance(model, data["dev_grad"], collate)
    Gn = G / (np.linalg.norm(G, axis=1, keepdims=True) + 1e-12)          # layer-normalised for global ranking
    prot_k, thr = kappa_protect(S["kappa"], KAPPA_Z)
    print("layer mean kappa:", np.round(S["kappa"].mean(1), 2), "| protected by rule:", prot_k, f"(thr={thr:.2f})")

    # are the spectral statistics a property of the PRETRAINED model or created by fine-tuning?
    if ("base", mname) not in cache:
        bm = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2).to(dev_).eval()
        cache[("base", mname)] = head_scores(bm)
        del bm
    Sb = cache[("base", mname)]
    sp_op = float(spearmanr(Sb["op"].ravel(), S["op"].ravel())[0])
    sp_kappa = float(spearmanr(Sb["kappa"].ravel(), S["kappa"].ravel())[0])
    print(f"Spearman(pretrained vs fine-tuned): op-norm={sp_op:.3f} kappa={sp_kappa:.3f}; "
          f"pretrained layer mean kappa={np.round(Sb['kappa'].mean(1), 2)}")

    rows, preds = [], {}

    def evaluate(method, k, draw, keep, extra=None):
        pr.apply(keep)
        lv, yv = predict(model, data["val"], collate)
        lt, yt = predict(model, data["train_sub"], collate)
        mv, mt = metrics(task, lv, yv), metrics(task, lt, yt)
        row = dict(model=mname, task=task, seed=seed, method=method, k=k, draw=draw,
                   n_kept=int(keep.sum()), frac_kept=float(keep.sum()) / (L * H),
                   kept_per_layer=" ".join(str(int(x)) for x in keep.sum(1)),
                   val_acc=mv["acc"], val_loss=mv["loss"], val_mcc=mv["mcc"], val_f1=mv["f1"],
                   train_acc=mt["acc"], train_loss=mt["loss"],
                   gap_loss=mv["loss"] - mt["loss"], gap_acc=mt["acc"] - mv["acc"],
                   C_fro=float(np.sqrt((S["frob"][keep] ** 2).sum())), C_op=float(S["op"][keep].sum()),
                   score_time_svd_s=S["_time_svd_s"], score_time_grad_s=t_grad)
        if extra: row.update(extra)
        rows.append(row)
        preds[f"{method}|{k}|{draw}"] = (lv.argmax(1)).astype(np.int8)
        if "labels" not in preds: preds["labels"] = yv.astype(np.int8)

    # dense reference
    evaluate("dense", H, 0, np.ones((L, H), bool))

    def want(name):
        return METHODS is None or name in METHODS

    for k in K_LIST:
        N = k * L
        if want("random_uniform"):
            for d in range(N_RANDOM):
                rs = np.random.RandomState(100000 * seed + 100 * k + d)
                evaluate("random_uniform", k, d, uniform_keep(rs.rand(L, H), k))
        if want("frob_uniform"):       evaluate("frob_uniform", k, 0, uniform_keep(S["frob"], k))
        if want("asicr_uniform"):      evaluate("asicr_uniform", k, 0, uniform_keep(S["op"], k))            # keep LARGEST operator norm
        if want("asicr_full_uniform"): evaluate("asicr_full_uniform", k, 0, uniform_keep(S["chain"], k))    # beta_Q*beta_K*beta_V*beta_O
        if want("grad_uniform"):       evaluate("grad_uniform", k, 0, uniform_keep(G, k))                   # Michel et al. style
        if want("asicr_global"):       evaluate("asicr_global", k, 0, global_keep(S["op"], N))
        if want("grad_global"):        evaluate("grad_global", k, 0, global_keep(Gn, N))
        # inverse criteria: keep the SMALLEST norms (= literally minimising the capacity proxy of Eq. 7)
        if want("asicr_inv_uniform"):  evaluate("asicr_inv_uniform", k, 0, uniform_keep(-S["op"], k))
        if want("frob_inv_uniform"):   evaluate("frob_inv_uniform", k, 0, uniform_keep(-S["frob"], k))
        if want("chain_inv_uniform"):  evaluate("chain_inv_uniform", k, 0, uniform_keep(-S["chain"], k))
        for name, P in [("la_kappa", prot_k), ("la_L0", [0])]:
            if not P or not want(f"asicr_{name}"):
                continue
            keep = la_keep(S["op"], k, P)
            if keep is not None:
                evaluate(f"asicr_{name}", k, 0, keep, {"protected": str(P)})
        print(f"  k={k} (N={N}) done")

    dloss = np.zeros((L, H))
    ld, yd = predict(model, data["dev_abl"], collate)
    base_loss = metrics(task, ld, yd)["loss"]
    for l in range(L if RUN_ABLATION else 0):
        for h in range(H):
            keep = np.ones((L, H), bool); keep[l, h] = False
            pr.apply(keep)
            lg, yy = predict(model, data["dev_abl"], collate)
            dloss[l, h] = metrics(task, lg, yy)["loss"] - base_loss        
    pr.apply(np.ones((L, H), bool))
    tau_rows = []
    for l in range(L if RUN_ABLATION else 0):
        t_imp, p_imp = kendalltau(S["op"][l], dloss[l])
        t_fr, p_fr = kendalltau(S["op"][l], S["frob"][l])
        tau_rows.append(dict(model=mname, task=task, seed=seed, layer=l,
                             tau_op_vs_ablation=t_imp, p_op_vs_ablation=p_imp,
                             tau_frob_vs_ablation=kendalltau(S["frob"][l], dloss[l])[0],
                             tau_op_vs_frob=t_fr, p_op_vs_frob=p_fr,
                             mean_kappa=S["kappa"][l].mean(), mean_kappa_row=S["kappa_row"][l].mean(),
                             mean_op=S["op"][l].mean()))
    rho_all = spearmanr(S["op"].ravel(), dloss.ravel())[0] if RUN_ABLATION else float("nan")
    print("  global Spearman(op norm, ablation dLoss) =", round(rho_all, 3))

    pd.DataFrame(rows).to_csv(f"{odir}/results.csv", index=False)
    if tau_rows:
        pd.DataFrame(tau_rows).to_csv(f"{odir}/layer_tau.csv", index=False)
    np.savez_compressed(f"{odir}/val_preds.npz", **preds)
    np.savez_compressed(f"{odir}/head_scores.npz", op=S["op"], frob=S["frob"], chain=S["chain"], kappa=S["kappa"],
                        kappa_row=S["kappa_row"], grad=G, ablation_dloss=dloss, spearman_all=rho_all)
    json.dump(dict(tag=tag, spearman_op_pretrained_vs_ft=sp_op, spearman_kappa_pretrained_vs_ft=sp_kappa,
                   base_layer_mean_kappa=[float(x) for x in Sb["kappa"].mean(1)],
                   ft_layer_mean_kappa=[float(x) for x in S["kappa"].mean(1)], protected_by_rule=prot_k, kappa_threshold=thr, svd_seconds=S["_time_svd_s"],
                   grad_seconds=t_grad, K_LIST=K_LIST, n_random=N_RANDOM), open(f"{odir}/meta.json", "w"), indent=2)
    open(f"{odir}/DONE", "w").write("ok")
    del model, pr
    torch.cuda.empty_cache()


if __name__ == "__main__" or True:
    runs = discover()
    print(f"Found {len(runs)} checkpoints:")
    for k, v in runs.items():
        print("  ", k, "->", v)
    assert runs, "No checkpoints found: add the fine-tuning notebook outputs via Add Input -> Notebook Output Files"
    cache = {}
    for key, path in runs.items():
        run_checkpoint(key, path, cache)
    allr = [pd.read_csv(p) for p in glob.glob(f"{OUT_DIR}/*/results.csv")]
    allt = [pd.read_csv(p) for p in glob.glob(f"{OUT_DIR}/*/layer_tau.csv")]
    if allr:
        pd.concat(allr).to_csv(f"{OUT_DIR}/ALL_results.csv", index=False)
    if allt:
        pd.concat(allt).to_csv(f"{OUT_DIR}/ALL_layer_tau.csv", index=False)
    print("\nALL DONE ->", OUT_DIR)

Found 10 checkpoints:
   ('distilbert-base-uncased', 'cola', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed0
   ('distilbert-base-uncased', 'cola', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed1
   ('distilbert-base-uncased', 'cola', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed2
   ('distilbert-base-uncased', 'cola', 3) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed3
   ('distilbert-base-uncased', 'cola', 4) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed4
   ('distilbert-base-uncased', 'mrpc', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_mrpc_seed0
   ('distilbert-base-uncased', 'mrpc', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_mrpc_seed1
   ('distilb

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

cola/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  251kB            

cola/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

cola/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.6kB            

cola/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (42 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_mrpc_seed3: L=6 H=12
layer mean kappa: [4.16 2.82 2.78 1.91 2.06 1.96] | protected by rule: [0] (thr=3.40)
Spearman(pretrained vs fine-tuned): op-norm=1.000 kappa=1.000; pretrained layer mean kappa=[4.16 2.82 2.78 1.91 2.06 1.96]
  k=10 (N=60) done
  k=8 (N=48) done
  k=6 (N=36) done
  k=4 (N=24) done
  k=3 (N=18) done
  k=2 (N=12) done
  global Spearman(op norm, ablation dLoss) = -0.094


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_mrpc_seed4: L=6 H=12
layer mean kappa: [4.16 2.82 2.78 1.91 2.06 1.96] | protected by rule: [0] (thr=3.40)
Spearman(pretrained vs fine-tuned): op-norm=1.000 kappa=1.000; pretrained layer mean kappa=[4.16 2.82 2.78 1.91 2.06 1.96]
  k=10 (N=60) done
  k=8 (N=48) done
  k=6 (N=36) done
  k=4 (N=24) done
  k=3 (N=18) done
  k=2 (N=12) done
  global Spearman(op norm, ablation dLoss) = -0.088

ALL DONE -> /kaggle/working/prune_results
